### ridge and Lesso Regression


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [4]:
df = pd.read_csv('housing_price_dataset.csv')

In [6]:
df.drop(columns=['YearBuilt'], inplace=True)

In [7]:
X = df.drop('Price', axis=1)
y = df['Price']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 
)

In [9]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = ['SquareFeet', 'Bedrooms', 'Bathrooms']
categorical_features = ['Neighborhood']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])


X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

In [10]:
print(feature_names)

['num__SquareFeet' 'num__Bedrooms' 'num__Bathrooms'
 'cat__Neighborhood_Rural' 'cat__Neighborhood_Suburb'
 'cat__Neighborhood_Urban']


In [11]:
ridge = Ridge()
ridge_params = {'alpha': [0.1, 1, 10, 100]}
ridge_cv = GridSearchCV(ridge, ridge_params, cv=5)
ridge_cv.fit(X_train, y_train)

print("Best Ridge alpha:", ridge_cv.best_params_)
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_test))))

Best Ridge alpha: {'alpha': 10}
Ridge RMSE: 49359.777975662255


In [12]:
lasso = Lasso(max_iter=10000)
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
lasso_cv = GridSearchCV(lasso, lasso_params, cv=5)
lasso_cv.fit(X_train, y_train)

print("Best Lasso alpha:", lasso_cv.best_params_)
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, lasso_cv.predict(X_test))))



Best Lasso alpha: {'alpha': 0.1}
Lasso RMSE: 49359.8047772267


In [13]:
# Inspect selected features from Lasso
lasso_coeffs = pd.Series(lasso_cv.best_estimator_.coef_, index=feature_names)
print("Lasso selected features (non-zero coefficients):")
print(lasso_coeffs[lasso_coeffs != 0])


Lasso selected features (non-zero coefficients):
num__SquareFeet             57145.890707
num__Bedrooms                5831.491320
num__Bathrooms               2421.846239
cat__Neighborhood_Rural      -742.075674
cat__Neighborhood_Suburb    -1338.729035
cat__Neighborhood_Urban       673.981390
dtype: float64
